# 第 40 章：Capstone 项目选题、范围控制与可执行性

这个 notebook 用一个极小的 proposal card 表，对比“只看 novelty / hype”的 naive baseline 和“先过数据 / 计算 / baseline / 交付边界 gate”的更可靠 scoping workflow。


In [ ]:
from __future__ import annotations

import csv
import platform
from collections import Counter
from pathlib import Path

DATA_PATH = Path('../../data/small/capstone_project_scoping_demo.csv')


def load_proposals(path):
    rows = []
    with path.open(encoding='utf-8', newline='') as handle:
        reader = csv.DictReader(handle)
        for row in reader:
            row['novelty_hype_score'] = float(row['novelty_hype_score'])
            rows.append(row)
    return rows


def ordered_counts(rows, field):
    counts = Counter(row[field] for row in rows)
    return {key: counts[key] for key in sorted(counts)}


rows = load_proposals(DATA_PATH)
print(f'Loaded {len(rows)} proposal cards from {DATA_PATH.name}')
print('Python version:', platform.python_version())
print('Science scope:', ordered_counts(rows, 'science_scope_status'))
print('Data access:', ordered_counts(rows, 'data_access_status'))
print('Compute fit:', ordered_counts(rows, 'compute_fit_status'))
print('Reference route:', ordered_counts(rows, 'reference_route'))


In [ ]:
reference_ready_ids = sorted(
    row['proposal_id']
    for row in rows
    if row['reference_route'] == 'ready_for_pilot'
)
top3_hype = sorted(
    rows,
    key=lambda row: row['novelty_hype_score'],
    reverse=True,
)[:3]
baseline_ready_hits = sum(
    row['reference_route'] == 'ready_for_pilot'
    for row in top3_hype
)
baseline_ready_precision = baseline_ready_hits / len(top3_hype)

print('Reference ready-for-pilot proposals:', reference_ready_ids)
print('Top-3 novelty / hype proposals:')
for row in top3_hype:
    print(
        f"  {row['proposal_id']}: hype={row['novelty_hype_score']:.2f}, "
        f"route={row['reference_route']}, theme={row['theme']}"
    )
print('Baseline pilot precision:', round(baseline_ready_precision, 3))


In [ ]:
def route_proposal(row):
    if row['data_access_status'] != 'public_ready':
        return 'blocked_data_boundary', 'data access or sharing boundary is not yet clear enough for a course capstone'
    if row['compute_fit_status'] != 'light' and row['baseline_feasibility'] == 'missing':
        return 'rethink_topic', 'the project is compute-heavy and still lacks even a minimal baseline plan'
    if row['deliverable_fit_status'] != 'fit':
        return 'rethink_topic', 'the expected notebook/report/showcase burden is too risky for the current course window'
    if row['science_scope_status'] != 'focused':
        return 'narrow_scope', 'the science question is still too broad for a one-semester capstone'
    if row['baseline_feasibility'] != 'ready':
        return 'narrow_scope', 'the topic still needs a simpler baseline before a larger model is justified'
    if row['literature_start_status'] != 'enough':
        return 'narrow_scope', 'the literature starting point is too thin for a grounded first pilot'
    return 'ready_for_pilot', 'the proposal is focused enough for a one-week pilot'


routed_rows = []
for row in rows:
    route, reason = route_proposal(row)
    routed = dict(row)
    routed['route'] = route
    routed['route_reason'] = reason
    routed_rows.append(routed)

exact_match_count = sum(
    row['route'] == row['reference_route']
    for row in routed_rows
)
workflow_accuracy = exact_match_count / len(routed_rows)

print('Workflow routing decisions:')
for row in routed_rows:
    print(
        f"  {row['proposal_id']}: predicted={row['route']}, "
        f"reference={row['reference_route']}, reason={row['route_reason']}"
    )
print('Exact routing accuracy:', round(workflow_accuracy, 3))


In [ ]:
ready_queue = [row for row in routed_rows if row['route'] == 'ready_for_pilot']
narrow_queue = [row for row in routed_rows if row['route'] == 'narrow_scope']
blocked_queue = [row for row in routed_rows if row['route'] == 'blocked_data_boundary']
rethink_queue = [row for row in routed_rows if row['route'] == 'rethink_topic']

ready_precision = sum(
    row['reference_route'] == 'ready_for_pilot'
    for row in ready_queue
) / max(len(ready_queue), 1)
ready_recall = sum(
    row['route'] == 'ready_for_pilot' and row['reference_route'] == 'ready_for_pilot'
    for row in routed_rows
) / max(len(reference_ready_ids), 1)

print('Ready-for-pilot queue:', [row['proposal_id'] for row in ready_queue])
print('Narrow-scope queue:', [row['proposal_id'] for row in narrow_queue])
print('Blocked-data-boundary queue:', [row['proposal_id'] for row in blocked_queue])
print('Rethink-topic queue:', [row['proposal_id'] for row in rethink_queue])
print('Structured-workflow pilot precision:', round(ready_precision, 3))
print('Structured-workflow pilot recall:', round(ready_recall, 3))


In [ ]:
def next_steps(row):
    steps = []
    if row['data_access_status'] != 'public_ready':
        steps.append('先明确数据是否公开、能否下载，以及是否允许进入课程仓库和 notebook。')
    if row['compute_fit_status'] != 'light':
        steps.append('把训练或分析规模缩到课程机器和时间窗口能够承受的版本。')
    if row['science_scope_status'] != 'focused':
        steps.append('把题目收窄成一个可测量、可展示的单一核心问题。')
    if row['baseline_feasibility'] != 'ready':
        steps.append('先写出一个一周内能跑通的 baseline，再谈更复杂模型。')
    if row['literature_start_status'] != 'enough':
        steps.append('先补 3 到 5 条 literature 起始卡片，确保 related work 和 caveat 有最小落点。')
    if row['deliverable_fit_status'] != 'fit':
        steps.append('重新估算 notebook、报告和展示负担，必要时直接换题。')
    return steps


def one_week_pilot(row):
    return [
        f"下载并清洗一个最小公开子集：{row['theme']}",
        '完成一个简单 baseline，并记录至少一个误差或失败样本。',
        '整理 3 到 5 条 literature ledger 起始卡片。',
        '写出一页 report outline 或 5 分钟展示草图。',
    ]


print('Next-step checklist for non-ready proposals:')
for row in routed_rows:
    if row['route'] == 'ready_for_pilot':
        continue
    print(f"\n{row['proposal_id']} -> {row['route']}")
    for step in next_steps(row):
        print(' -', step)

pilot_plan = one_week_pilot(ready_queue[0])
print('\nOne-week pilot plan for the first ready proposal:')
for step in pilot_plan:
    print(' -', step)


In [ ]:
scoping_assistant_prompt = '''你是我的 capstone 选题助手。

任务：
1. 先阅读 proposal card，而不是先推荐更大的模型；
2. 依次检查题目范围、数据获取、计算负担、baseline 可行性、
   literature 起点和交付风险；
3. 把每个题目 route 到 ready_for_pilot、narrow_scope、
   blocked_data_boundary 或 rethink_topic；
4. 只有在 route 明确后，才给下一周 pilot checklist。

输出格式：
- Ready-for-pilot proposals
- Narrow-scope proposals
- Blocked-data-boundary proposals
- Rethink-topic proposals
- One-week pilot checklist
'''
print(scoping_assistant_prompt)


In [ ]:
workflow_checklist = [
    '先问数据和 scope，而不是先问模型名够不够新。',
    '没有一周 pilot 的题目，通常还不适合直接做期末主线。',
    'baseline、literature 起点和交付边界都要在选题阶段写清楚。',
    'narrow_scope 往往是健康出口，不是项目失败。',
    '题目太大时，尽早换题比期末前临时补救要安全得多。',
]

proposal_snapshot = {
    'dataset': DATA_PATH.name,
    'n_proposals': len(rows),
    'baseline_top3_pilot_precision': round(baseline_ready_precision, 3),
    'workflow_route_accuracy': round(workflow_accuracy, 3),
    'ready_for_pilot': [row['proposal_id'] for row in ready_queue],
    'narrow_scope': [row['proposal_id'] for row in narrow_queue],
    'blocked_data_boundary': [row['proposal_id'] for row in blocked_queue],
    'python_version': platform.python_version(),
}

print('Workflow checklist:')
for item in workflow_checklist:
    print(' -', item)

print('\nProposal snapshot:')
for key, value in proposal_snapshot.items():
    print(f'  {key}: {value}')
